# Глава 15. Масштабирование и производительность

## 15.1. Оптимизация производительности RAG

**Зачем всё это нужно**

Для корпоративных RAG‑систем важны скорость ответа, пропускная способность и стоимость запросов. Они напрямую влияют на удобство пользователя и расходы на эксплуатацию.

**Что оптимизировать в первую очередь**

- Скорость токенизации.
- Эффективность энкодеров.
- Стратегию поиска по векторной базе.
- Интеграцию с языковой моделью (LLM).

**Векторное хранилище**

Выбор и настройка векторного хранилища критически важны. Решения вроде HNSW, FAISS (особенно GPU‑версия), DiskANN или Milvus ускоряют поиск по большому индексу в разы. На практике задержка поиска падает с секунд до миллисекунд даже при миллионах документов.

**Индексация и подготовка данных**

Их нужно делать с расчётом на быстрый поиск:
- подбирать размер сегментов и пересечения;
- объединять тематически связанные фрагменты;
- удалять дубликаты чанков.

Это уменьшает объём бесполезного контекста и снижает нагрузку на генерацию.

**Сжатие и квантизация эмбеддингов**

Это эффективный способ сэкономить память и ускорить поиск. Форматы float8, int8 или эмбеддинги, сжатые через PCA, дают ускорение и снижение стоимости хранения при потере качества меньше 0,5%. В крупных пайплайнах это лучше сочетать с уменьшением размера индекса — точность почти не падает.

## 15.2. Горизонтальное и вертикальное масштабирования

**Масштабирование RAG-систем** — это переход от прототипа к рабочей системе под большой нагрузкой. Есть два основных подхода:

**1. Вертикальное масштабирование** — усиление одного сервера: больше ядер, памяти, быстрые диски, GPU.  
Плюсы: просто управлять, не нужно менять архитектуру.  
Минус: упирается в предел одного сервера.

**2. Горизонтальное масштабирование** — добавление новых серверов в кластер.  
Позволяет распределить нагрузку между:
- репликами векторной базы,
- балансировщиками,
- сервисами эмбеддингов,
- инстансами языковых моделей.

**Как это работает:**

- Векторное хранилище делится на **шарды** (части), часто автоматически.
- Запросы идут к наименее загруженным узлам.
- Если один узел падает, система продолжает работать.
- Данные можно размещать в разных регионах — это снижает задержки и повышает надёжность.

**Автомасштабирование** — система сама добавляет или убирает ресурсы по метрикам: CPU, память, длина очередей.

**Сегментирование данных** даёт почти неограниченную масштабируемость: запросы идут только к нужным шардам, результаты собираются и передаются в языковую модель.

**Лучший вариант — гибрид:**  

- критичные компоненты масштабируются вертикально,  
- легко распараллеливаемые сервисы — горизонтально.

**Сложный случай:** графовые структуры хуже масштабируются горизонтально, потому что нужно обходить связи между узлами. Тут помогают гибридные подходы — графы + векторы.